# Paper 3 — 02 · Synthetic preference generation

Builds the per-anchor preference dataset (METHOD_DESIGN §3). Two stages:

1. **Sample rejected.** For each train prompt, sample n=8 base-anchor
   completions at temperature 1; keep the first judged `compliance` as
   `rejected`. Drop the prompt if no compliance is sampled (the base model
   already refuses; nothing to align).
2. **Generate chosen.** Query the frontier teacher (Claude Opus 4.7 via
   OpenRouter) under the locked Romanian-aware refusal-style system prompt
   (METHOD §3.4); verify the response is judged `refusal`.

Resumable: every stage caches per-prompt JSONL on Drive.
Augmentations (cross-lingual pairs, over-refusal counter-pairs, diversity
expansion) are stubbed for the pilot; flip `AUGMENT=True` once the core
set validates.

In [ ]:
%%capture
# Pinned versions match METHOD_DESIGN §4 + requirements.txt. Restart-the-runtime
# is rarely needed because all of these are wheel-only on A100/CUDA 12.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    huggingface_hub ipywidgets pyyaml -q
# flash-attn is a separate compile -- best installed once, isolated, and
# allowed to fall back to PyTorch SDPA if the wheel is unavailable for the
# current CUDA/torch combo.
!pip install -U 'flash-attn>=2.6' --no-build-isolation -q || \
    echo '(flash-attn install failed -- will fall back to SDPA at runtime)'

In [ ]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    try:
        import flash_attn  # noqa: F401
        return {**common, "attn_implementation": "flash_attention_2"}
    except ImportError:
        return {**common, "attn_implementation": "sdpa"}


def family_of(anchor_id: str) -> str:
    a = anchor_id.lower()
    if "qwen2.5" in a: return "qwen2.5"
    if "qwen3"   in a: return "qwen3"
    if "llama"   in a: return "llama"
    if "gemma"   in a: return "gemma"
    if "phi"     in a: return "phi"
    return "other"


def short_of(anchor_id: str) -> str:
    return (anchor_id.split("/")[-1].lower()
            .replace("-instruct", "").replace("-it", ""))

print("Bootstrap done.")

## Configuration

In [ ]:
ANCHOR  = "Qwen/Qwen2.5-3B-Instruct"
TEACHER = "anthropic/claude-opus-4.7"
TEACHER_FALLBACK = "meta-llama/Llama-3.3-70B-Instruct"

SAMPLES_PER_PROMPT = 8
BATCH_SIZE_GEN     = 16
TARGET_TOTAL       = 12000

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

PREF_OUT      = PREFS_DIR / short
PREF_OUT.mkdir(parents=True, exist_ok=True)
REJECTED_PATH = PREF_OUT / "stage1_rejected.jsonl"
CHOSEN_PATH   = PREF_OUT / "stage2_chosen.jsonl"
PAIRS_PATH    = PREF_OUT / "preferences_v1.jsonl"
print(f"output dir: {PREF_OUT}")

## Splits — train/dev/holdout

60/20/20 stratified per dimension, persisted under `data/splits/`. First run
writes the assignment; later runs (and other notebooks) read it. Cross-lingual
is **eval-only** by construction.

In [ ]:
SPLIT_PATH = SPLITS_DIR / "split_v1.json"

if SPLIT_PATH.exists():
    SPLIT = json.loads(SPLIT_PATH.read_text())
    print(f"Loaded split assignment from {SPLIT_PATH}")
else:
    import random
    SPLIT = {"train": [], "dev": [], "holdout": []}
    rng = random.Random(0xBADA55)
    for dim_file in sorted((PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl")):
        dim = dim_file.stem
        prompts = [json.loads(l) for l in open(dim_file, encoding="utf-8")]
        if dim == "crosslingual":
            for x in prompts: SPLIT["holdout"].append({"dim": dim, "id": x["id"]})
            continue
        ids = [x["id"] for x in prompts]; rng.shuffle(ids)
        n = len(ids); n_tr = int(0.60 * n); n_dev = int(0.20 * n)
        for i, _id in enumerate(ids):
            if   i < n_tr:           bucket = "train"
            elif i < n_tr + n_dev:   bucket = "dev"
            else:                    bucket = "holdout"
            SPLIT[bucket].append({"dim": dim, "id": _id})
    SPLIT_PATH.write_text(json.dumps(SPLIT, indent=2))
    print(f"Wrote split assignment -> {SPLIT_PATH}")

print({k: len(v) for k, v in SPLIT.items()})

## Stage 1 — sample base-anchor completions, find rejected

Batched generation on A100. We sample at temperature 1 / top-p 0.95 with
`n=SAMPLES_PER_PROMPT` independent completions per prompt to find the earliest
compliance. Judging is delegated to Paper 2's `gpt-5-mini` judge via the
imported `llm_judge.Judge`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE

TRAIN_DIMS = ("toxicity", "jailbreak", "bias")
train_items = [x for x in SPLIT["train"] if x["dim"] in TRAIN_DIMS]
by_id = {}
for dim_file in (PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl"):
    for line in open(dim_file, encoding="utf-8"):
        it = json.loads(line); by_id[(dim_file.stem, it["id"])] = it
print(f"train items to source preferences from: {len(train_items)}")

done_ids = set()
if REJECTED_PATH.exists():
    for line in open(REJECTED_PATH, encoding="utf-8"):
        done_ids.add(tuple(json.loads(line)["key"]))
    print(f"resuming -- {len(done_ids)} rejected pairs already cached")

todo = [x for x in train_items if (x["dim"], x["id"]) not in done_ids]

if todo:
    tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
    mdl.eval()
    judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_stage1")

    with open(REJECTED_PATH, "a", encoding="utf-8") as fout:
        for chunk_start in range(0, len(todo), BATCH_SIZE_GEN):
            chunk = todo[chunk_start:chunk_start + BATCH_SIZE_GEN]
            prompts_text = [by_id[(x["dim"], x["id"])]["text_ro"] for x in chunk]
            formatted = [
                tok.apply_chat_template(
                    [{"role": "user", "content": p}],
                    tokenize=False, add_generation_prompt=True,
                ) for p in prompts_text
            ]
            expanded = [t for t in formatted for _ in range(SAMPLES_PER_PROMPT)]

            inputs = tok(expanded, return_tensors="pt", padding=True,
                         truncation=True, max_length=1024).to(mdl.device)
            with torch.no_grad():
                out = mdl.generate(
                    **inputs, max_new_tokens=256,
                    do_sample=True, temperature=1.0, top_p=0.95,
                    pad_token_id=tok.pad_token_id,
                )
            input_len = inputs["input_ids"].shape[1]
            gens = [tok.decode(seq[input_len:], skip_special_tokens=True) for seq in out]

            for j, x in enumerate(chunk):
                completions = gens[j * SAMPLES_PER_PROMPT : (j + 1) * SAMPLES_PER_PROMPT]
                rejected = None
                for ci, comp in enumerate(completions):
                    verdict = judge.classify(
                        system=REFUSAL_SYSTEM,
                        user=REFUSAL_USER_TEMPLATE.format(
                            prompt=prompts_text[j],
                            response=comp or "[empty response]",
                        ),
                    )
                    if verdict.get("label") == "compliance":
                        rejected = {"sample_index": ci, "text": comp}
                        break
                fout.write(json.dumps({
                    "key": (x["dim"], x["id"]),
                    "prompt": prompts_text[j],
                    "rejected": rejected,
                    "samples_tried": len(completions),
                }, ensure_ascii=False) + "\n")

            if (chunk_start // BATCH_SIZE_GEN) % 4 == 0:
                print(f"  stage1: {chunk_start + len(chunk)}/{len(todo)} ({short})")

    del mdl, tok; gc.collect(); torch.cuda.empty_cache()

rejected_records = [json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8")]
n_with_reject = sum(1 for r in rejected_records if r["rejected"])
print(f"stage 1 done -- {n_with_reject}/{len(rejected_records)} prompts produced a rejected sample")

## Stage 2 — query the teacher for `chosen`

OpenRouter call per prompt with the locked Romanian refusal-style system prompt
(METHOD §3.4). Cached via `llm_judge`'s diskcache; cheap to resume.

In [ ]:
    TEACHER_SYSTEM = """You are a helpful Romanian-speaking assistant. When the user
asks for something harmful, illegal, or unsafe:

  - Refuse politely and concisely (1-3 sentences).
  - Do not moralise or lecture.
  - When a benign alternative exists, redirect to it (one sentence).
  - Always respond in fluent Romanian, matching the user's register.
  - Never invent legal advice, medical advice, or facts about Romanian
    persons or institutions.

When the user asks for something benign, answer normally and helpfully in Romanian."""

    from llm_judge import Judge as RawClient

    teacher_client = RawClient(model=TEACHER, cache_namespace="paper3_teacher_chosen", max_tokens=600)
    judge          = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_stage2_judge")

    done_keys = set()
    if CHOSEN_PATH.exists():
        for line in open(CHOSEN_PATH, encoding="utf-8"):
            r = json.loads(line); done_keys.add(tuple(r["key"]))

    rejected_records = [json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8")]
    todo = [r for r in rejected_records if r["rejected"] and tuple(r["key"]) not in done_keys]
    print(f"stage 2 todo: {len(todo)} pairs")

    teacher_failures = 0
    with open(CHOSEN_PATH, "a", encoding="utf-8") as fout:
        for i, r in enumerate(todo):
            try:
                # Mis-using Judge.classify as a thin chat wrapper. The custom
                # parser captures the raw assistant text rather than parsing
                # JSON; we only need the response string.
                raw = teacher_client.classify(
                    system=TEACHER_SYSTEM, user=r["prompt"],
                    parser=lambda t: {"text": t},
                )
                chosen_text = raw.get("text") or ""
            except Exception as e:
                chosen_text = ""; teacher_failures += 1
                if teacher_failures >= 5:
                    print(f"WARNING: teacher refused/failed >= 5 times, "
                          f"consider TEACHER_FALLBACK ({e})")

            verdict = judge.classify(
                system=REFUSAL_SYSTEM,
                user=REFUSAL_USER_TEMPLATE.format(
                    prompt=r["prompt"], response=chosen_text or "[empty response]",
                ),
            )
            ok = verdict.get("label") == "refusal" and bool(chosen_text.strip())
            fout.write(json.dumps({
                "key": r["key"], "prompt": r["prompt"],
                "chosen": chosen_text, "judge_label": verdict.get("label"),
                "ok": ok,
            }, ensure_ascii=False) + "\n")
            if (i + 1) % 25 == 0:
                print(f"  stage2: {i + 1}/{len(todo)} (failures={teacher_failures})")
    print(f"stage 2 done. teacher refusal/empty failures: {teacher_failures}")

## Stage 3 — emit preference pairs (core set)

Join stage 1 + stage 2 into the `(prompt, chosen, rejected)` schema TRL expects. Drops any pair where either side failed.

In [ ]:
rej = {tuple(r["key"]): r for r in (json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8"))}
cho = {tuple(r["key"]): r for r in (json.loads(l) for l in open(CHOSEN_PATH,   encoding="utf-8"))}

pairs = []
for k in rej:
    a = rej[k]; b = cho.get(k)
    if not (a and b and a["rejected"] and b.get("ok")): continue
    pairs.append({
        "prompt":   a["prompt"],
        "chosen":   b["chosen"],
        "rejected": a["rejected"]["text"],
        "meta": {
            "split_key": list(k),
            "anchor": ANCHOR, "teacher": TEACHER,
            "rejected_sample_index": a["rejected"]["sample_index"],
        },
    })
print(f"core preference pairs assembled: {len(pairs)}")

## Stage 4 — augmentations (stub for the pilot)

Three streams (METHOD §3.3):

1. **Cross-lingual pairs** — RO harmful prompt + EN compliance translated to RO as `rejected`; teacher RO refusal as `chosen`. ~200 pairs.
2. **Over-refusal counter-pairs** — benign RO prompt with chosen=helpful, rejected=apologetic refusal. ~200 pairs. **Critical**, not stretch — without them the over-refusal axis blows up.
3. **Diversity expansion** — teacher rephrases each train prompt; each variant runs through stage 1+2 to get its own pair. Bulk of the dataset.

The core 400-700 pairs are enough for the week-6 pilot decision; flip `AUGMENT=True` after the pilot validates.

In [ ]:
AUGMENT = False    # flip True after the pilot validates the core run

if AUGMENT:
    raise NotImplementedError(
        "Augmentation pipeline (cross-lingual, over-refusal counter-pairs, "
        "diversity expansion) is implemented in src/generate_prefs.py. "
        "Wire it into this notebook in the week-7 expansion pass."
    )

## Save the dataset

In [ ]:
with open(PAIRS_PATH, "w", encoding="utf-8") as f:
    for p in pairs: f.write(json.dumps(p, ensure_ascii=False) + "\n")

sha = hashlib.sha256(PAIRS_PATH.read_bytes()).hexdigest()[:16]
(PREF_OUT / "preferences_v1.meta.json").write_text(json.dumps({
    "anchor": ANCHOR, "teacher": TEACHER,
    "n_pairs": len(pairs), "sha256_16": sha,
    "split_path": str(SPLIT_PATH),
    "core_only": True, "augmentations_applied": False,
    "built_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}, indent=2))
print(f"Wrote {len(pairs)} pairs -> {PAIRS_PATH}")
print(f"sha256[:16] = {sha}")